# Imports

In [1]:
%reload_ext autoreload
%autoreload 2


from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.evaluate_mmlu import evaluate_mmlu
sys.path.pop(0);

/home/oh/arubinstein17/github/disco-public/envs/disco_env_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Evaluate new model

## Get ground truth

### Take from leaderboard snapshot

In [2]:
df = pd.read_csv(os.path.join(ROOT_PATH, "benchmark_csvs","open-llm-leaderboard.csv"))
small_models = df[df['#Params (B)'] <= 8]
small_models.head(n=10)


,T,Model,Average ⬆️,ARC,HellaSwag,MMLU,TruthfulQA,Winogrande,GSM8K,Type,...,Weight type,Precision,Merged,Hub License,#Params (B),Hub ❤️,Available on the hub,Model sha,Flagged,MoE
14,🔶,alnrg2arg/test2_4,75.28,73.55,88.87,64.63,69.77,84.45,70.43,fine-tuned,...,Original,bfloat16,True,apache-2.0,7.24,0.0,False,ed17cf5af87733ffd7836ab99f27991544ba2547,False,False
16,🟦,udkai/Garrulus,75.16,73.29,88.87,64.57,68.23,91.48,64.52,RL-tuned,...,Original,float16,False,apache-2.0,7.24,10.0,True,cd2fa5c2188588b903fff2070a389db3b24031a4,True,False
18,🟦,eren23/slerp-test-turdus-beagle,75.11,73.55,88.85,64.62,69.69,83.90,70.05,RL-tuned,...,Original,bfloat16,True,apache-2.0,7.24,0.0,True,f2aef36538bb0c7aab30ffe889e12b72f51a6816,False,False
23,?,rwitz2/go-bruins-v2.1.1,74.95,72.87,88.33,65.18,69.80,82.24,71.27,NaN,...,Original,bfloat16,False,cc,7.24,22.0,True,bd56295eab54eaacbb3af6ecb88b9434d9966d4e,True,False
25,🔶,nfaheem/Marcoroni-7b-DPO-Merge,74.90,73.04,88.80,64.24,70.47,85.24,67.63,fine-tuned,...,Original,float16,True,apache-2.0,7.24,1.0,True,e3085d8aacffbf46b95e263bde509fce70577a26,False,False
26,🔶,quantumaikr/quantum-dpo-v0.1,74.87,72.53,88.37,65.29,69.92,82.32,70.81,fine-tuned,...,Original,float16,False,cc-by-nc-4.0,7.24,2.0,True,09cbfe6569bcdddf623e9990498e9ad07345ad6a,True,False
27,🔶,jan-hq/trinity-v1,74.80,72.27,88.36,65.20,69.31,82.00,71.65,fine-tuned,...,Original,bfloat16,True,apache-2.0,7.24,11.0,True,34974ae99668c381be0871778e3c42958544f70e,True,False
28,🔶,janai-hq/trinity-v1,74.80,72.27,88.36,65.20,69.31,82.00,71.65,fine-tuned,...,Original,bfloat16,True,apache-2.0,7.24,4.0,True,09da1a24f84c96b8c09f2c07038986e28cc24ad5,True,False
29,🔶,mlabonne/Beagle14-7B,74.76,72.95,87.95,64.70,68.88,82.64,71.42,fine-tuned,...,Original,bfloat16,True,apache-2.0,7.24,8.0,True,a5d1b1f831efe38df3b6ac125764a87ed094e282,False,False
30,🔶,alnrg2arg/test2_3,74.76,72.95,88.42,64.80,68.40,84.14,69.83,fine-tuned,...,Original,bfloat16,True,apache-2.0,7.24,0.0,True,e2c681fa4680ee19ca9758a2289da7d168546672,False,False


In [3]:
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
except:
    print("transformers not installed; run `pip install transformers`")


model_id = "alnrg2arg/test2_4"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.53s/it]


### Eval on the whole dataset to compute ground truth from scratch

In [ ]:
evaluate_mmlu(
    model_id=model_id,
    subset="all",
    num_fewshot=5,
    max_samples=None,
    batch_size=4,
)

## DISCO-eval

In [ ]:
# get anchor points
# get predictor
# get outputs for anchor points, cache
# predict with predictor
# compare to ground truth

# Debug

In [2]:
bench = "mmlu_fields"
data, scenarios, set_of_rows, data_path = load_and_split_model_outputs(
    bench=bench,
    split='noniid',
    model_outputs_path=MODEL_OUTPUTS_PATH,
    text_to_vector=None,
    return_data_path=True,
    subsample_validation=False,
)

chosen_scenarios = list(scenarios.keys())
split_number = 0
rows_to_hide = set_of_rows[split_number]
iterations = 5


40 425


In [3]:
(
    scores_train,
    predictions_train,
    predictions_test,
    scores_test,
    balance_weights,
    scenarios_position,
    subscenarios_position,
    rows_to_hide_per_iteration
) = prepare_and_split_data(
    chosen_scenarios,
    scenarios,
    data,
    rows_to_hide=rows_to_hide,
    n_source_models=None,
    iterations=iterations,
    # different_split_per_iteration=False,
    different_split_per_iteration=True,
    bootstrap_ratio=0.90,
)

In [4]:
sampling_names = ["high-disagreement"]
# sampling_names = ["high-disagreement+nonstratified"]
disagreement_type = "pds"
disagreement_scores_dict = make_disagreement_scores_dict(
    config={
        "sampling_names": sampling_names,
        "predictions_train": predictions_train,
        "disagreement_type": disagreement_type,
        "iterations": iterations,
    }
)

In [5]:

number_items = [100]
# iterations = 5
sampling_name = sampling_names[0]
number_item = number_items[0]
seen_items_dic = {sampling_name: {}}
apply_random_seed(RANDOM_SEED)
samples = sample_items(
    number_item,
    iterations,
    sampling_name,
    chosen_scenarios,
    scenarios,
    subscenarios_position,
    responses_test=scores_test,
    scores_train=scores_train,
    predictions_train=predictions_train,
    scenarios_position=scenarios_position,
    A=None,
    B=None,
    balance_weights=balance_weights,
    disagreement_scores_dict=disagreement_scores_dict,
    skip_irt=True,
)
(
    _,
    seen_items_dic[sampling_name][number_item],
    _,
    _
) = samples

In [6]:
pca = 256
# pca = None
apply_softmax_to_predictions = True
(
    train_models_embeddings,
    test_models_embeddings,
) = make_train_test_model_embeddings(
    emb_config={
        "sampling_names": sampling_names,
        "number_items": number_items,
        "iterations": iterations,
        "predictions_train": predictions_train,
        "seen_items_dic": seen_items_dic,
        "predictions_test": predictions_test,
        "seen_items_dic": seen_items_dic,
        "pca": pca,
        "apply_softmax": apply_softmax_to_predictions,
    }
)


Computing embeddings: 100%|██████████| 1/1 [00:04<00:00,  4.08s/it]


In [7]:
train_model_indices = {}
train_model_true_accs = {}
for it in range(iterations):
    train_model_indices[it] = list(range(scores_train[it].shape[0]))
    train_model_true_accs[it] = compute_true_acc(
        scores_train[it],
        balance_weights,  # sample -> sample weight
        scenarios_position,  # scenario -> list of sample indices
        chosen_scenarios,
        train_model_indices[it],
        train_model_indices[it],  # they are not the global indices, but the contiguous indices of train models after removing test models
    )

In [ ]:

fitted_model_type = "RandomForestRegressor_100"
chosen_fitting_methods = [
    (fitted_model_type, (sklearn.ensemble.RandomForestRegressor, {"n_estimators": 100}))
]
scenario = bench.split("_")[0]
fitted_weights = make_fitted_weights(
    config={
        "sampling_names": sampling_names,
        "number_items": number_items,
        "iterations": iterations,
        "train_models_embeddings": train_models_embeddings,
        "train_model_true_accs": train_model_true_accs,
        "scenario": scenario,
        "cache_path": "./cache_for_notebook",
        "chosen_fitting_methods": chosen_fitting_methods,
        "skip_iterations_when_fixed_sampling": False,
    }
)

Training RMSE of RandomForestRegressor_100 for samplng=high-disagreement, n_anchor=100, run=0: 0.012203984824112217
Training RMSE of RandomForestRegressor_100 for samplng=high-disagreement, n_anchor=100, run=1: 0.011555277116466402


In [ ]:
predictors_per_seed = fitted_weights[sampling_names[0]][100]
target_model_embeddings_per_seed = test_models_embeddings[sampling_names[0]][100]
predicted_accs = {}
predicted_accs_knn = {}
for seed in range(iterations):
    predicted_accs[seed] = {}
    predicted_accs_knn[seed] = {}
    fitted_model = predictors_per_seed[seed][fitted_model_type]
    target_model_embeddings = target_model_embeddings_per_seed[seed]
    for target_model_idx in range(target_model_embeddings.shape[0]):
        test_model_embedding = target_model_embeddings[target_model_idx]
        fitted_acc = fitted_model.predict(
            test_model_embedding.numpy().reshape(1, -1)
        )[0]
        fitted_acc_knn = compute_acc_knn(
            # test_model_embedding=test_model_embedding[seed],
            test_model_embedding=test_model_embedding,
            train_model_embeddings=train_models_embeddings[
                sampling_name
            ][number_item][seed],
            scenario=scenario,
            train_model_true_accs=train_model_true_accs[seed],
        )
        # target_model = rows_to_hide[target_model_idx]
        target_model = rows_to_hide_per_iteration[seed][target_model_idx]
        if target_model not in predicted_accs[seed]:
            predicted_accs[seed][target_model] = []
            predicted_accs_knn[seed][target_model] = []
        predicted_accs[seed][target_model].append(fitted_acc)
        predicted_accs_knn[seed][target_model].append(fitted_acc_knn)

In [ ]:
scores = load_scores(
    bench,
    split='noniid',
    scenarios_to_skip=[],
    ordered=True,
    filename_suffix="",
    num_it=5,
    data_path=None,
    filter_indices_by_results=False
)
# ?? use corresponding test indices for each split
# gt_scores = {}
# for it in range(iterations):
# gt_scores = np.stack([scores[:, rows_to_hide_per_iteration[it]] for it in range(iterations)], axis=2)
gt_scores = {}
for seed in range(iterations):
    gt_scores[seed] = {}
    for target_model in rows_to_hide_per_iteration[seed]:
        gt_scores[seed][target_model] = scores[:, target_model]

In [ ]:
from torch.xpu.random import seed_all


maes_per_method = {}
rank_corrs_per_method = {}
predictions = {
    "fitted": predicted_accs,
    "knn": predicted_accs_knn
}
for seed in range(iterations):
    maes_per_method[seed] = {}
    rank_corrs_per_method[seed] = {}
    for method_name, accs in predictions.items():
        for target_model in accs[seed].keys():
            rank_corrs = []
            # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
            # pred_accs_as_np = np.stack(list(accs.values()), axis=0)
            # print(pred_accs_as_np.shape)
            # maes = np.abs(
            #     # pred_accs_as_np - gt_scores[0][:, None]
            #     pred_accs_as_np[seed] - gt_scores[0][:, seed]
            # )
            mae = np.abs(accs[seed][target_model] - gt_scores[seed][target_model])
            # for i in range(pred_accs_as_np.shape[1]):
            #     rank_corrs.append(safe_spearmanr(
            #         pred_accs_as_np[:, i],
            #         # gt_scores[0][:, None],
            #         gt_scores[0]
            #     ))
            # rank_corrs.append(safe_spearmanr(
            #     accs[seed],
            #     # gt_scores[0][:, None],
            #     gt_scores[seed]
            # ))
            # print(accs[seed].shape)
            # print(gt_scores[seed].shape)
            model_names_list = list(accs[seed].keys())
            accs_list = list(accs[seed][model_name] for model_name in model_names_list)
            gt_scores_list = [gt_scores[seed][model_name] for model_name in model_names_list]
            rank_corrs_per_method[seed][method_name] = safe_spearmanr(
                accs_list,
                # gt_scores[0][:, None],
                gt_scores_list
            )
            if method_name not in maes_per_method[seed]:
                maes_per_method[seed][method_name] = {}
            if target_model not in maes_per_method[seed][method_name]:
                maes_per_method[seed][method_name][target_model] = []
            maes_per_method[seed][method_name][target_model].append(mae)
            # rank_corrs_per_method[method_name] = np.array(rank_corrs)

In [ ]:
maes_mean = {}
maes_std = {}
for method_name in maes_per_method[0].keys():
    maes_mean[method_name] = {}
    maes_std[method_name] = {}
    for target_model in maes_per_method[0][method_name].keys():
        per_model_maes = []
        for seed in range(iterations):
            maes = maes_per_method[seed][method_name].get(target_model, [])
            # print(maes)
            assert len(maes) <= 1
            if len(maes) == 1:
                per_model_maes.append(maes[0])


        maes_mean[method_name][target_model] = (np.array(per_model_maes)).mean(axis=0)
        maes_std[method_name][target_model] = (np.array(per_model_maes)).std(axis=0)

mae_mean = {}
mae_std = {}
for method_name in maes_mean.keys():
    maes_means = np.array([maes_mean[method_name][target_model] for target_model in maes_mean[method_name].keys()])
    maes_stds = np.array([maes_std[method_name][target_model] for target_model in maes_std[method_name].keys()])

    mae_mean[method_name] = maes_means.mean(axis=0)[0]
    mae_std[method_name] = maes_stds.mean(axis=0)[0]


In [ ]:
print(mae_mean)
print(mae_std)


{'fitted': np.float64(0.013846605170314427), 'knn': np.float64(0.01715105294691259)}
{'fitted': np.float64(0.005953756240720465), 'knn': np.float64(0.004992161419548962)}


In [ ]:
for method_name, maes in maes_per_method[0].items():
    # maes = np.array([maes_per_method[seed][method_name] for seed in range(iterations)])
    # print(maes.shape)
    rank_corrs = np.array([rank_corrs_per_method[seed][method_name] for seed in range(iterations)])
    # print(rank_corrs)
    # mae_str = f"{maes.mean(axis=1).mean()} +- {maes.std(axis=0).mean()}"
    mae_str = f"{mae_mean[method_name]} +- {mae_std[method_name]}"

    rank_corrs_str = f"{rank_corrs.mean().mean()} +- {rank_corrs.std()}"
    # rank_corrs_str = ""
    print(f"{method_name}: {mae_str} | {rank_corrs_str}")

fitted: 0.013846605170314427 +- 0.005953756240720465 | 0.9611840411840411 +- 0.006041837102831788
knn: 0.01715105294691259 +- 0.004992161419548962 | 0.9395202554126177 +- 0.010399346767308382
